# Step 1: Install Libraries

In [ ]:
!pip install pymupdf
!pip install sentence-transformers
!pip install supabase
!pip install langchain-text-splitters
!pip install groq

In [ ]:
!pip install vecs

In [ ]:
import os
import fitz
import torch

from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from supabase import create_client
from groq import Groq

In [ ]:
from supabase import create_client, Client

# Step 2: Supabase Connection and Setup

In [ ]:
SUPABASE_URL = "YOUR_SUPABASE_URL"
SUPABASE_KEY = "YOUR_SUPABASE_KEY"

supabase: Client = create_client(
    SUPABASE_URL,
    SUPABASE_KEY
)

In [ ]:
client = Groq(
    api_key="YOUR_GROQ_API_KEY"
)

#Step 3: Loading Embedding Model

In [ ]:
model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

print("Embedding model loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded


# Step 4: PDF Text Extraction

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
PDF_FOLDER = "/PATH/TO/DRIVE/DATASET/FOLDER/"

In [ ]:
def extract_text_from_pdf(pdf_path):

    doc = fitz.open(pdf_path)

    pages = []

    for page_number, page in enumerate(doc):

        text = page.get_text()

        pages.append({
            "page_number": page_number + 1,
            "text": text
        })

    doc.close()

    return pages

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

NOTE: The below table is made usually in Supabase SQL Editor for production-grade RAG.

In [ ]:
# ============================================================
# STEP 10 — CREATE EMBEDDINGS TABLE
# ============================================================

# RUN THIS SQL IN SUPABASE SQL EDITOR
"""
create extension if not exists vector;

create table if not exists document_chunks (

    id bigserial primary key,

    file_name text,

    storage_path text,

    page_number int,

    chunk_index int,

    chunk_text text,

    embedding vector(384)

);
"""


# ============================================================
# STEP 11 — CREATE CHAT MEMORY TABLE
# ============================================================

# RUN THIS SQL IN SUPABASE SQL EDITOR
"""
create table if not exists chat_memory (

    id bigserial primary key,

    session_id text,

    role text,

    message text,

    created_at timestamp default now()

);
"""


# ============================================================
# STEP 12 — CREATE MATCH FUNCTION
# ============================================================

# RUN THIS SQL IN SUPABASE SQL EDITOR
"""
create or replace function match_documents (
    query_embedding vector(384),
    match_count int default 5
)

returns table (
    id bigint,
    file_name text,
    chunk_text text,
    similarity float
)

language sql
as $$

select
    id,
    file_name,
    chunk_text,

    1 - (embedding <=> query_embedding) as similarity

from document_chunks

order by embedding <=> query_embedding

limit match_count;

$$;
"""


'\ncreate or replace function match_documents (\n    query_embedding vector(384),\n    match_count int default 5\n)\n\nreturns table (\n    id bigint,\n    file_name text,\n    chunk_text text,\n    similarity float\n)\n\nlanguage sql\nas $$\n\nselect\n    id,\n    file_name,\n    chunk_text,\n\n    1 - (embedding <=> query_embedding) as similarity\n\nfrom document_chunks\n\norder by embedding <=> query_embedding\n\nlimit match_count;\n\n$$;\n'

In [ ]:
print(os.listdir(PDF_FOLDER))

['vm-import-ug.pdf', 'ec2-types.pdf', 'ec2-instance-connect-api.pdf', 'sql-server-ec2.pdf', 'enclaves-user.pdf', 'ec2-dg.pdf']


In [ ]:
import vecs

DB_CONNECTION = "SUPABASE_KEY"

# create vector store client
vx = vecs.create_client(DB_CONNECTION)
docs = vx.get_or_create_collection(name="TABLE_NAME", dimension=384)

In [ ]:
from supabase import create_client, Client

# Step 5: Chunking

In [ ]:
# ============================================================
# STEP 13 — CREATE MATCH FUNCTION
# ============================================================
import os
all_rows = []

for file_name in os.listdir(PDF_FOLDER):

    if file_name.endswith(".pdf"):

        print(f"Processing: {file_name}")

        file_path = os.path.join(
            PDF_FOLDER,
            file_name
        )

        # ====================================================
        # UPLOAD PDF TO SUPABASE STORAGE
        # ====================================================

        with open(file_path, "rb") as f:

            try:

                supabase.storage \
                    .from_("TABLE_NAME") \
                    .upload(
                        f"pdfs/{file_name}",
                        f
                    )

                print(f"Uploaded {file_name}")

            except Exception as e:

                print("Already exists or upload issue")
                print(e)

        # ====================================================
        # EXTRACT PDF TEXT
        # ====================================================

        pages = extract_text_from_pdf(file_path)

        # ====================================================
        # CHUNK EACH PAGE
        # ====================================================

        for page in pages:

            chunks = splitter.split_text(
                page["text"]
            )

            for chunk_index, chunk in enumerate(chunks):

                all_rows.append({

                    "file_name": file_name,

                    "storage_path":
                    f"pdfs/{file_name}",

                    "page_number":
                    page["page_number"],

                    "chunk_index":
                    chunk_index,

                    "chunk_text":
                    chunk
                })


print(f"Total chunks: {len(all_rows)}")


Processing: vm-import-ug.pdf
Already exists or upload issue
'dict' object has no attribute 'text'
Processing: ec2-types.pdf
Already exists or upload issue
'dict' object has no attribute 'text'
Processing: ec2-instance-connect-api.pdf
Already exists or upload issue
'dict' object has no attribute 'text'
Processing: sql-server-ec2.pdf
Already exists or upload issue
'dict' object has no attribute 'text'
Processing: enclaves-user.pdf
Already exists or upload issue
'dict' object has no attribute 'text'
Processing: ec2-dg.pdf
Already exists or upload issue
'dict' object has no attribute 'text'
Total chunks: 9071


In [ ]:
all_rows

[{'file_name': 'vm-import-ug.pdf',
  'storage_path': 'pdfs/vm-import-ug.pdf',
  'page_number': 1,
  'chunk_index': 0,
  'chunk_text': 'User Guide\nVM Import/Export\nCopyright © 2026 Amazon Web Services, Inc. and/or its aﬃliates. All rights reserved.'},
 {'file_name': 'vm-import-ug.pdf',
  'storage_path': 'pdfs/vm-import-ug.pdf',
  'page_number': 2,
  'chunk_index': 0,
  'chunk_text': "VM Import/Export\nUser Guide\nVM Import/Export: User Guide\nCopyright © 2026 Amazon Web Services, Inc. and/or its aﬃliates. All rights reserved.\nAmazon's trademarks and trade dress may not be used in connection with any product or service \nthat is not Amazon's, in any manner that is likely to cause confusion among customers, or in any \nmanner that disparages or discredits Amazon. All other trademarks not owned by Amazon are"},
 {'file_name': 'vm-import-ug.pdf',
  'storage_path': 'pdfs/vm-import-ug.pdf',
  'page_number': 2,
  'chunk_index': 1,
  'chunk_text': 'the property of their respective owners, wh

In [ ]:
# ============================================================
# STEP 14 — GENERATE EMBEDDINGS
# ============================================================

texts = [
    row["chunk_text"]
    for row in all_rows
]

embeddings = model.encode(
    texts,
    batch_size=16,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Embeddings created")


Batches:   0%|          | 0/567 [00:00<?, ?it/s]

Embeddings created


# Step 6: Storing Embeddings to Supabase

In [ ]:
# ============================================================
# STEP 15 — INSERT INTO SUPABASE
# ============================================================

insert_rows = []

for i, row in enumerate(all_rows):

    insert_rows.append({

        "file_name":
        row["file_name"],

        "storage_path":
        row["storage_path"],

        "page_number":
        row["page_number"],

        "chunk_index":
        row["chunk_index"],

        "chunk_text":
        row["chunk_text"],

        "embedding":
        embeddings[i].tolist()
    })

# ============================================================
# INSERT CHUNKS
# ============================================================

response = supabase.table(
    "db1_table"
).insert(
    insert_rows
).execute()

print("Inserted into Supabase")

APIError: {'message': 'JSON could not be generated', 'code': 401, 'hint': 'Refer to full message for details', 'details': 'b\'{"message":"Invalid API key","hint":"Double check your Supabase `anon` or `service_role` API key."}\''}

#Step 7: Chat Loop

In [ ]:
# ============================================================
# STEP 16 — CHAT LOOP
# ============================================================

SESSION_ID = "session_1"

while True:

    # ========================================================
    # USER QUERY
    # ========================================================

    query = input("\nAsk Question: ")

    if query.lower() == "exit":
        break

    # ========================================================
    # LOAD PREVIOUS MEMORY
    # ========================================================

    memory_response = supabase.table(
        "chat_memory"
    ).select(
        "*"
    ).eq(
        "session_id",
        SESSION_ID
    ).order(
        "created_at"
    ).execute()

    # ========================================================
    # BUILD CHAT HISTORY
    # ========================================================

    chat_context = ""

    for row in memory_response.data:

        chat_context += (
            f"{row['role']}: "
            f"{row['message']}\n"
        )

    # ========================================================
    # CREATE QUERY EMBEDDING
    # ========================================================

    query_embedding = model.encode(
        query,
        normalize_embeddings=True
    ).tolist()

    # ========================================================
    # RETRIEVE RELEVANT DOCUMENTS
    # ========================================================

    retrieval = supabase.rpc(
        "match_documents",
        {
            "query_embedding": query_embedding,
            "match_count": 5
        }
    ).execute()

    # ========================================================
    # BUILD DOCUMENT CONTEXT
    # ========================================================

    document_context = ""

    for row in retrieval.data:

        document_context += (
            f"\n[FILE: {row['file_name']}]\n"
            f"{row['chunk_text']}\n"
        )

    # ========================================================
    # PROMPT TEMPLATE
    # ========================================================

    prompt = f"""
You are a helpful AI assistant.

Use the previous conversation and retrieved
documents to answer the question.

If answer is not present in documents,
say you do not know.

===============================
PREVIOUS CONVERSATION
===============================

{chat_context}

===============================
RETRIEVED DOCUMENTS
===============================

{document_context}

===============================
CURRENT QUESTION
===============================

{query}

===============================
ANSWER
===============================
"""

    # ========================================================
    # LLM CALL
    # ========================================================

    response = client.chat.completions.create(

        model="llama-3.3-70b-versatile",

        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    answer = response.choices[0].message.content

    # ========================================================
    # PRINT ANSWER
    # ========================================================

    print("\n============================")
    print("ANSWER")
    print("============================\n")

    print(answer)

    # ========================================================
    # STORE MEMORY
    # ========================================================

    supabase.table(
        "chat_memory"
    ).insert([

        {
            "session_id": SESSION_ID,
            "role": "user",
            "message": query
        },

        {
            "session_id": SESSION_ID,
            "role": "assistant",
            "message": answer
        }

    ]).execute()

    print("\nMemory Updated")